In [ ]:
%load_ext autoreload
%autoreload 2

# Histograms of Model Predictions (Test Set)

Compute and plot histograms of predicted values for all models on the test set,
compared against the ground-truth observations and raw IFS ensemble.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from genpp.data.weatherbench2 import (
    FC_VARS,
    FORECAST_ENS_PATH,
    OBSERVATIONS_FLAT_PATH,
    TEST_PREDICTIONS,
)
from genpp.eval import best_models
from genpp.eval.utils import load_predictions_dataarray
from genpp.plots import RESULTS_DIR

## Load Observations and Raw Ensemble

In [ ]:
# Ground-truth observations
valid_times = (
    TEST_PREDICTIONS.get_level_values("time")
    + TEST_PREDICTIONS.get_level_values("prediction_timedelta")
)
obs = (
    xr.open_dataset(OBSERVATIONS_FLAT_PATH)[FC_VARS]
    .sel(time=valid_times)
    .to_dataarray("feature")
    .sel(feature=FC_VARS)
    .transpose("time", "feature", "longitude", "latitude")
)
print("Observations shape:", obs.shape)

# Raw IFS ensemble
raw = (
    xr.open_dataset(FORECAST_ENS_PATH)[FC_VARS]
    .sel(
        time=TEST_PREDICTIONS.get_level_values("time").unique(),
        prediction_timedelta=TEST_PREDICTIONS.get_level_values("prediction_timedelta").unique(),
    )
    .stack(prediction=["time", "prediction_timedelta"])
    .sel(prediction=TEST_PREDICTIONS)
    .rename({"number": "sample"})
    .to_dataarray("feature")
    .transpose("prediction", "sample", "feature", "latitude", "longitude")
)
print("Raw ensemble shape:", raw.shape)

## Load Model Predictions

In [ ]:
# Discover and load test-set predictions for all models
model_predictions: dict[str, dict[str, xr.DataArray]] = {}

for model_name, model_entries in best_models:
    if not model_entries:
        continue
    model_predictions[model_name] = {}

    # For EMOS and DRN: only load ECC and GCA copula variants
    if model_name in ("emos", "drn"):
        entry = model_entries[0]
        for copula_tag, copula_file in [
            ("ecc", "test_predictions_ecc.zarr"),
            ("gca", "test_predictions_gca.zarr"),
        ]:
            try:
                copula_path = entry.model_dir / copula_file
                if not copula_path.exists():
                    found = list(entry.model_dir.rglob(copula_file))
                    if not found:
                        print(f"  ✗ {model_name}/{copula_tag}: {copula_file} not found")
                        continue
                    copula_path = found[0]
                preds = load_predictions_dataarray(copula_path).sel(prediction=TEST_PREDICTIONS)
                model_predictions[model_name][copula_tag] = preds
                n_samples = preds.sizes.get("sample", 1)
                print(f"  ✓ {model_name}/{copula_tag}: {preds.shape}  ({n_samples} samples)")
            except Exception as e:
                print(f"  ✗ {model_name}/{copula_tag}: {e}")
    else:
        # All other models: load test_predictions variants
        for entry in model_entries:
            variant_key = entry.tag or "standard"
            try:
                pred_files = list(entry.model_dir.rglob("test_predictions*.zarr"))
                if not pred_files:
                    print(f"  ✗ {model_name}/{variant_key}: no prediction file found")
                    continue
                preds = load_predictions_dataarray(pred_files[0]).sel(prediction=TEST_PREDICTIONS)
                model_predictions[model_name][variant_key] = preds
                n_samples = preds.sizes.get("sample", 1)
                print(f"  ✓ {model_name}/{variant_key}: {preds.shape}  ({n_samples} samples)")
            except Exception as e:
                print(f"  ✗ {model_name}/{variant_key}: {e}")

print(f"\nLoaded models: {list(model_predictions.keys())}")
for mname, variants in model_predictions.items():
    print(f"  {mname}: {list(variants.keys())}")

## Plot Histograms per Variable

For each forecast variable we plot a histogram of the observed values alongside
histograms of the raw ensemble and every post-processed model variant.
All sample and spatial dimensions are flattened before binning.

In [ ]:
VAR_DISPLAY = {
    "2m_temperature": "2m Temperature (K)",
    "10m_wind_speed": "10m Wind Speed (m/s)",
}

N_BINS = 80
HIST_ALPHA = 0.5

for var in FC_VARS:
    display_name = VAR_DISPLAY.get(var, var)

    # Observed values (flatten all spatial and time dims)
    obs_vals = obs.sel(feature=var).values.ravel()

    # Raw ensemble values (flatten sample + spatial + time dims)
    raw_vals = raw.sel(feature=var).values.ravel()

    # Determine common bin edges from the observations
    lo = float(np.nanmin(obs_vals))
    hi = float(np.nanmax(obs_vals))
    pad = 0.05 * (hi - lo)
    bins = np.linspace(lo - pad, hi + pad, N_BINS + 1)

    # Collect all model variants into a flat list for plotting
    model_items: list[tuple[str, np.ndarray]] = []
    for model_name, variants in model_predictions.items():
        for variant_key, da in variants.items():
            label = f"{model_name}/{variant_key}"
            vals = da.sel(feature=var).values.ravel()
            model_items.append((label, vals))

    # --- Figure ---
    n_models = len(model_items)
    n_cols = 4
    # Rows: 1 for obs+raw, then ceil(n_models / n_cols) for model panels
    n_model_rows = max(1, int(np.ceil(n_models / n_cols)))
    n_rows = 1 + n_model_rows

    fig, axes = plt.subplots(
        n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows), constrained_layout=True
    )
    axes = np.atleast_2d(axes)
    fig.suptitle(f"Histograms – {display_name}", fontsize=16, fontweight="bold")

    # -- Row 0: Observations and Raw Ensemble --
    ax_obs = axes[0, 0]
    ax_obs.hist(obs_vals, bins=bins, density=True, alpha=HIST_ALPHA, color="black", label="Obs")
    ax_obs.set_title("Observations")
    ax_obs.set_ylabel("Density")
    ax_obs.legend()

    ax_raw = axes[0, 1]
    ax_raw.hist(raw_vals, bins=bins, density=True, alpha=HIST_ALPHA, color="tab:blue", label="Raw")
    ax_raw.hist(obs_vals, bins=bins, density=True, histtype="step", color="black", linewidth=1.0, label="Obs")
    ax_raw.set_title("Raw Ensemble")
    ax_raw.legend()

    # Hide unused axes in row 0
    for j in range(2, n_cols):
        axes[0, j].set_visible(False)

    # -- Remaining rows: one panel per model variant --
    for idx, (label, vals) in enumerate(model_items):
        row = 1 + idx // n_cols
        col = idx % n_cols
        ax = axes[row, col]
        ax.hist(vals, bins=bins, density=True, alpha=HIST_ALPHA, color="tab:orange", label=label)
        ax.hist(obs_vals, bins=bins, density=True, histtype="step", color="black", linewidth=1.0, label="Obs")
        ax.set_title(label, fontsize=10)
        ax.legend(fontsize=7)

    # Hide leftover axes
    for idx in range(n_models, n_model_rows * n_cols):
        row = 1 + idx // n_cols
        col = idx % n_cols
        axes[row, col].set_visible(False)

    plt.savefig(RESULTS_DIR / f"histograms_{var}.pdf", bbox_inches="tight")
    plt.show()

## Overlay Histograms

A single overlay histogram per variable showing one representative variant per model
family for a quick comparison.

In [ ]:
# Select one representative variant per model family
REPRESENTATIVE_VARIANTS = {
    "emos": "ecc",
    "drn": "ecc",
    "chen": "ind_es",
    "engression": "ind_es",
    "fm": "ind_unet",
}

COLORS = {
    "Obs": "black",
    "Raw": "tab:blue",
    "emos": "tab:orange",
    "drn": "tab:green",
    "chen": "tab:red",
    "engression": "tab:purple",
    "fm": "tab:brown",
}

MODEL_DISPLAY = {
    "emos": "EMOS",
    "drn": "DRN",
    "chen": "LNGM",
    "engression": "Engression",
    "fm": "FM",
}

for var in FC_VARS:
    display_name = VAR_DISPLAY.get(var, var)

    obs_vals = obs.sel(feature=var).values.ravel()
    raw_vals = raw.sel(feature=var).values.ravel()

    lo = float(np.nanmin(obs_vals))
    hi = float(np.nanmax(obs_vals))
    pad = 0.05 * (hi - lo)
    bins = np.linspace(lo - pad, hi + pad, N_BINS + 1)

    fig, ax = plt.subplots(figsize=(10, 5))

    ax.hist(obs_vals, bins=bins, density=True, histtype="step", color=COLORS["Obs"],
            linewidth=2.0, label="Obs")
    ax.hist(raw_vals, bins=bins, density=True, histtype="step", color=COLORS["Raw"],
            linewidth=1.5, linestyle="--", label="Raw Ensemble")

    for model_name, variant_key in REPRESENTATIVE_VARIANTS.items():
        if model_name not in model_predictions:
            continue
        if variant_key not in model_predictions[model_name]:
            # Fall back to first available variant
            available = list(model_predictions[model_name].keys())
            if not available:
                continue
            variant_key = available[0]
        da = model_predictions[model_name][variant_key]
        vals = da.sel(feature=var).values.ravel()
        label = MODEL_DISPLAY.get(model_name, model_name)
        ax.hist(vals, bins=bins, density=True, histtype="step",
                color=COLORS.get(model_name, None), linewidth=1.2, label=label)

    ax.set_title(f"Histogram Comparison – {display_name}", fontsize=14)
    ax.set_xlabel(display_name)
    ax.set_ylabel("Density")
    ax.legend()

    plt.savefig(RESULTS_DIR / f"histograms_overlay_{var}.pdf", bbox_inches="tight")
    plt.show()